<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Ming-3-TTS_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Ming-3-TTS_Colab.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-View_Source-181717?logo=github" alt="View on GitHub"/></a>

# 🎙️ Ming 3.0 TTS (`Ming-3.0-tts-0.5B` Dense & `Ming-3.0-tts-tiny` MoE) — 44.1 kHz Studio Speech Synthesis, Zero-Shot Voice Cloning, Voice Design, 482 Built-in Voices & Podcast Studio

Run **inclusionAI's Ming 3.0 TTS** ([`inclusionAI/Ming-3.0-tts-0.5B`](https://huggingface.co/inclusionAI/Ming-3.0-tts-0.5B) & [`inclusionAI/Ming-3.0-tts-tiny`](https://huggingface.co/inclusionAI/Ming-3.0-tts-tiny) · [GitHub](https://github.com/inclusionAI/Ming-3.0-tts) · [Space Demo](https://huggingface.co/spaces/hugging-apps/ming-3-tts-05b)) on Google Colab with persistent Google Drive caching and a 5-tab Gradio Studio UI.

On the **Artificial Analysis Speech Arena Leaderboards (October 2026)**, Ming 3.0 TTS ranks **#1 among open-weight models on Controlled Voice** (Arena Elo 1069), **#1 overall on Pronunciation Robustness** (93.2% overall, 96.5% shorthand normalization), and **#3 among open-weight models on Provider Voice** (Arena Elo 1188).

### ✨ Key Capabilities (44.1 kHz Output)
- **Zero-Shot Voice Cloning (`TTS` / `MULTILINGUAL`)** — Clone any 3–10s reference clip across **7 languages** (`Chinese`, `English`, `Japanese`, `Korean`, `German`, `French`, `Spanish`) using dual 192-d CampPlus speaker embeddings + 12.5 Hz AudioVAE latent continuation, with **1-click CPU Whisper auto-transcription** for custom reference clips.
- **14 Inline Paralinguistic Tags** — Insert natural vocal expressions anywhere in Chinese or English text: `[Laughter]`, `[Sigh]`, `[Breathing]`, `[Cough]`, `[Uhm]`, `[Surprise-ah]`, `[Surprise-oh]`, `[Dissatisfaction-hnn]`, `[Confirmation-en]`, `[Question-ei]`, `[Question-ah]`, `[Question-en]`, `[Question-oh]`, `[Question-huh]`.
- **Voice Design (`VOICE_DESIGN`) & Free-Form Delivery (`FREEFORM`)** — Design brand-new voices from text persona descriptions (including a **12-attribute structured voice sheet builder**) or guide speaking style with natural-language instructions, and **save designed voices directly to your Drive Voice Library** to clone across all 7 languages or in podcasts.
- **482 Built-in Premium Voices (`IP`)** — 5 English voices (`US_James`, `US_Stella`, `US_Emma`, `UK_George`, `UK_Charlotte`), 48 Chinese base voices (`灵小柔`, `灵楷`, `灵小安`, etc.), and 429 Chinese emotion-specific variants (`灵惜晨_高兴`, `灵阅川_悲伤`, etc.) with live persona role cards.
- **Explicit Emotion, Dialect & Prosody Attribute Control (`EMOTION`, `DIALECT`, `BASIC`)** — Control 7 emotions (`高兴/happy`, `悲伤/sad`, `愤怒/angry`, `恐惧/fearful`, `厌恶/disgusted`, `惊讶/surprised`, `中性/neutral`), **Cantonese / Sichuanese** dialects, or **Speed / Pitch / Volume** while preserving reference speaker timbre.
- **Two-Speaker Podcast Studio (`podcast`)** — Generate multi-turn `speaker_1:` / `speaker_2:` dialogues via native `Ming-3.0-tts-tiny` multi-speaker conditioning or turn-by-turn stitched synthesis on `Ming-3.0-tts-0.5B`.

| Model Variant | HF Repo ID | Architecture | Disk / VRAM | Recommended Colab GPU | License |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **`Ming-3.0-tts-0.5B`** *(Default)* | `inclusionAI/Ming-3.0-tts-0.5B` | Dense (`Qwen2.5-0.5B` + Flow-Matching DiT + 44.1 kHz AudioVAE) | **2.80 GB** / **~4 GB VRAM** | **T4 (16 GB)**, **L4 (24 GB)**, **A100 (40 GB)** | **MIT** |
| **`Ming-3.0-tts-tiny`** *(Flagship MoE)* | `inclusionAI/Ming-3.0-tts-tiny` | MoE (`Ling V3` 7.9B total · 1.3B active + Flow-Matching DiT) | **16.77 GB** / **~19 GB VRAM** | **L4 (24 GB)**, **A100 (40 GB)** | **MIT** |

> **Cross-Notebook Avatar Integration**: All generated `.wav` clips are automatically saved to `/content/drive/MyDrive/AEI_TTS_Out/Ming-3-TTS/`, where `InfiniteTalk_Colab.ipynb`, `LongCat-Video-Avatar-1.5_Colab.ipynb`, and `Seed-VC_Colab.ipynb` automatically discover them via their Cross-Notebook Drive Audio Picker.

In [ ]:
#@title Step 1: Verify GPU & Install Ming 3.0 TTS Environment
import os
import sys
import json
import shutil
import subprocess
import urllib.request
import importlib.util

try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "No CUDA GPU detected! In Colab, go to Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4, L4, or A100)."
        )
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    print(f"GPU detected: {gpu_name} | VRAM: {vram_gb:.1f} GB | Compute Capability: sm_{cc_major}{cc_minor}")
    if vram_gb < 22.0:
        print("Tier recommendation: Use 'Ming-3.0-tts-0.5B (Dense)' (~4 GB VRAM) on T4 16GB.")
    else:
        print("Tier recommendation: Both 'Ming-3.0-tts-0.5B (Dense)' and 'Ming-3.0-tts-tiny (MoE 7.9B)' will fit in VRAM.")
except Exception as e:
    print(f"GPU check warning: {e}")
    raise

REPO_DIR = "/content/Ming-3.0-tts"

try:
    if not os.path.exists(REPO_DIR):
        print("Cloning inclusionAI/Ming-3.0-tts repository...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/inclusionAI/Ming-3.0-tts.git", REPO_DIR],
            check=True,
        )
    else:
        print("Updating existing Ming-3.0-tts clone...")
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

    # Fetch the 482-voice catalog (voices.json) from the official Hugging Face Space
    voices_json_path = os.path.join(REPO_DIR, "voices.json")
    if not os.path.exists(voices_json_path):
        print("Fetching 482 built-in voice catalog (voices.json) from hugging-apps/ming-3-tts-05b...")
        req = urllib.request.Request(
            "https://huggingface.co/spaces/hugging-apps/ming-3-tts-05b/raw/main/voices.json",
            headers={"User-Agent": "Mozilla/5.0"},
        )
        with urllib.request.urlopen(req, timeout=20) as resp:
            with open(voices_json_path, "wb") as vf:
                vf.write(resp.read())

    print("Installing Ming 3.0 TTS Python dependencies...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "accelerate", "safetensors", "soundfile", "scipy", "tqdm", "loguru",
            "x_transformers==2.9.2", "diffusers", "onnxruntime", "inflect",
            "einops", "einx==0.4.3", "fla-core==0.4.2", "packaging", "gradio>=4.44.0",
            "huggingface_hub>=0.25.0",
        ],
        check=True,
    )

    # Install pynini for FST text normalization (numbers, dates, units, symbols) with graceful fallback
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "pynini==2.1.6.post1"],
            check=True,
        )
        print("Installed pynini==2.1.6.post1 for FST text normalization.")
    except Exception as pyn_err:
        print(f"Optional pynini install skipped ({pyn_err}); pipeline will run with text_normalization=False fallback.")

    # Install transformers==4.46.3 + tokenizers==0.20.3 (--no-deps) so BOTH Ming-3.0-tts-0.5B
    # and Ming-3.0-tts-tiny run in the same runtime without huggingface_hub<1.0 conflicts.
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            "transformers==4.46.3", "tokenizers==0.20.3",
        ],
        check=True,
    )
    spec = importlib.util.find_spec("transformers")
    if spec and spec.submodule_search_locations:
        dep_table = os.path.join(spec.submodule_search_locations[0], "dependency_versions_table.py")
        if os.path.exists(dep_table):
            with open(dep_table, "r", encoding="utf-8") as f:
                src = f.read()
            patched = src.replace('"huggingface-hub>=0.23.2,<1.0"', '"huggingface-hub>=0.23.2"')
            if patched != src:
                with open(dep_table, "w", encoding="utf-8") as f:
                    f.write(patched)

    # Apply the Space KV-cache causal mask patch to modeling_bailingmm.py
    # and Turing T4 (sm_75) SDPA safety to configuration_bailingmm.py
    bailing_model_py = os.path.join(REPO_DIR, "modeling_bailingmm.py")
    if os.path.exists(bailing_model_py):
        with open(bailing_model_py, "r", encoding="utf-8") as f:
            bsrc = f.read()
        old_call = "outputs = self.model(\n                    attention_mask=attention_mask,"
        new_call = "outputs = self.model(\n                    attention_mask=None,"
        if old_call in bsrc:
            with open(bailing_model_py, "w", encoding="utf-8") as f:
                f.write(bsrc.replace(old_call, new_call))
            print("Patched modeling_bailingmm.py (attention_mask=None on cached decode steps).")

    bailing_cfg_py = os.path.join(REPO_DIR, "configuration_bailingmm.py")
    if os.path.exists(bailing_cfg_py):
        with open(bailing_cfg_py, "r", encoding="utf-8") as f:
            csrc = f.read()
        old_check = 'if importlib.util.find_spec("flash_attn") is not None:\n        return config'
        new_check = (
            'import torch\n'
            '    has_ampere = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8\n'
            '    if importlib.util.find_spec("flash_attn") is not None and has_ampere:\n'
            '        return config'
        )
        if old_check in csrc:
            with open(bailing_cfg_py, "w", encoding="utf-8") as f:
                f.write(csrc.replace(old_check, new_check))
            print("Patched configuration_bailingmm.py (force SDPA on Turing/T4 GPUs).")

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print("Ming 3.0 TTS environment setup complete!")
except Exception as e:
    print(f"Error during Step 1 installation: {e}")
    raise


In [ ]:
#@title Step 2: Mount Google Drive & Pre-Cache Ming 3.0 TTS Checkpoints
import os
import sys
import subprocess

USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
PRECACHE_05B_DENSE = True #@param {type:"boolean"}
PRECACHE_TINY_MOE = False #@param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        DRIVE_BASE = "/content/drive/MyDrive"
    except Exception as e:
        print(f"Google Drive mount unavailable ({e}); falling back to /content/local_drive.")
        DRIVE_BASE = "/content/local_drive"
else:
    DRIVE_BASE = "/content/local_drive"

CACHE_ROOT = os.path.join(DRIVE_BASE, "AEI_TTS_Cache")
HF_CACHE_DIR = os.path.join(CACHE_ROOT, "huggingface")
MODELS_DIR = os.path.join(CACHE_ROOT, "models")
VOICES_LIB_DIR = os.path.join(CACHE_ROOT, "voices")
OUTPUT_DIR = os.path.join(DRIVE_BASE, "AEI_TTS_Out", "Ming-3-TTS")

for d in (CACHE_ROOT, HF_CACHE_DIR, MODELS_DIR, VOICES_LIB_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["TORCH_HOME"] = os.path.join(CACHE_ROOT, "torch")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Configure portable SDPA attention for Ming-3.0-tts-tiny AudioVAE & MoE
os.environ.setdefault("MING_TINY_AUDIO_VAE_ATTN", "sdpa")
os.environ.setdefault("BAILING_MLA_ATTN", "sdpa")
os.environ.setdefault("BAILING_KDA_FUSED_RECURRENT", "0")
os.environ.setdefault("FLA_USE_CUDA_KDA_FWD", "0")
os.environ.setdefault("FLA_USE_CUDA_KDA_BWD", "0")

MODEL_05B_REPO = "inclusionAI/Ming-3.0-tts-0.5B"
MODEL_TINY_REPO = "inclusionAI/Ming-3.0-tts-tiny"
MODEL_05B_DIR = os.path.join(MODELS_DIR, "Ming-3.0-tts-0.5B")
MODEL_TINY_DIR = os.path.join(MODELS_DIR, "Ming-3.0-tts-tiny")

try:
    from huggingface_hub import snapshot_download

    if PRECACHE_05B_DENSE:
        print(f"Downloading / verifying {MODEL_05B_REPO} (~2.80 GB) -> {MODEL_05B_DIR} ...")
        snapshot_download(
            repo_id=MODEL_05B_REPO,
            local_dir=MODEL_05B_DIR,
            ignore_patterns=["*.pdf", "figures/*"],
        )
        preflight_py = "/content/Ming-3.0-tts/tools/preflight.py"
        if os.path.exists(preflight_py):
            subprocess.run([sys.executable, preflight_py, "--model_path", MODEL_05B_DIR], check=False)
        print("Ming-3.0-tts-0.5B checkpoint verified on Drive!")

    if PRECACHE_TINY_MOE:
        print(f"Downloading / verifying {MODEL_TINY_REPO} (~16.77 GB) -> {MODEL_TINY_DIR} ...")
        snapshot_download(
            repo_id=MODEL_TINY_REPO,
            local_dir=MODEL_TINY_DIR,
            ignore_patterns=["*.pdf", "figures/*"],
        )
        preflight_tiny_py = "/content/Ming-3.0-tts/tools/preflight_tiny.py"
        if os.path.exists(preflight_tiny_py):
            subprocess.run([sys.executable, preflight_tiny_py, "--model_path", MODEL_TINY_DIR], check=False)
        print("Ming-3.0-tts-tiny MoE checkpoint verified on Drive!")

    print(f"Cache root : {CACHE_ROOT}")
    print(f"Voices lib : {VOICES_LIB_DIR}")
    print(f"Output dir : {OUTPUT_DIR}")
except Exception as e:
    print(f"Error during model download/caching: {e}")
    raise


In [ ]:
#@title Step 3: Initialize Unified Ming 3.0 TTS Studio Engine (0.5B Dense & Tiny MoE)
import os
import sys
import re
import gc
import time
import json
import glob
import shutil
import random
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import numpy as np
import soundfile as sf
import torch
import torchaudio

REPO_DIR = "/content/Ming-3.0-tts"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import tts_tasks
from ming_tts import MingTTS, trim_trailing_silence

# Check optional pynini availability
try:
    import pynini  # noqa: F401
    HAS_PYNINI = True
except Exception:
    HAS_PYNINI = False

# Patch MingTTS._load_prompt to use soundfile.read directly (avoids torchaudio>=2.9 torchcodec errors)
def _safe_load_prompt(self, paths):
    waveforms, spk_emb = [], []
    for path in paths:
        data, sample_rate = sf.read(path, dtype="float32", always_2d=True)
        if data.size == 0:
            raise ValueError(f"Prompt audio is empty: {path}")
        waveform = torch.from_numpy(data.T.copy()).float()
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        speaker_waveform = waveform
        if sample_rate != 16000:
            speaker_waveform = torchaudio.transforms.Resample(sample_rate, 16000)(waveform)
        spk_emb.append(self.spkemb_extractor(speaker_waveform))

        if sample_rate != self.sample_rate:
            waveform = torchaudio.transforms.Resample(sample_rate, self.sample_rate)(waveform)
        waveforms.append(waveform)

    waveform = torch.cat(waveforms, dim=-1)
    if hasattr(self, "_pad_prompt_waveform"):
        return self._pad_prompt_waveform(waveform), spk_emb
    align = self._pad_align()
    remainder = waveform.shape[-1] % align
    if remainder:
        padded = torch.zeros(
            (waveform.shape[0], waveform.shape[-1] + align - remainder),
            dtype=waveform.dtype,
        )
        padded[:, -waveform.shape[-1]:] = waveform
        waveform = padded
    return waveform, spk_emb

MingTTS._load_prompt = _safe_load_prompt

# Load built-in voice catalog (482 voices across 3 categories)
VOICES_JSON_PATH = os.path.join(REPO_DIR, "voices.json")
if os.path.exists(VOICES_JSON_PATH):
    with open(VOICES_JSON_PATH, "r", encoding="utf-8") as vf:
        VOICE_GROUPS: Dict[str, List[str]] = json.load(vf)
else:
    VOICE_GROUPS = {
        "English": ["US_James", "US_Stella", "US_Emma", "UK_George", "UK_Charlotte"],
        "Chinese": ["灵小柔", "灵楷", "灵小安", "灵语汐", "灵小萌", "灵小雅", "花小呗", "灵韵"],
        "Chinese (emotion variants)": ["灵惜晨_高兴", "灵惜晨_悲伤", "灵阅川_中性", "灵阅川_高兴", "灵未央_中性"],
    }
ALL_VOICES: List[str] = [v for grp in VOICE_GROUPS.values() for v in grp]

# Curated persona & role descriptions from inclusionAI's official cookbook
VOICE_PERSONA_CATALOG: Dict[str, str] = {
    "US_James": "🇺🇸 English (US) · Male · Warm, articulate American narrator suited for audiobooks, explainers & podcasts.",
    "US_Stella": "🇺🇸 English (US) · Female · Bright, friendly American voice suited for customer service, assistants & narration.",
    "US_Emma": "🇺🇸 English (US) · Female · Natural, conversational American voice for product dialogue, assistants & storytelling.",
    "UK_George": "🇬🇧 English (UK) · Male · Composed British baritone for formal announcements, documentaries & audiobooks.",
    "UK_Charlotte": "🇬🇧 English (UK) · Female · Measured, refined British narrator for classic literature & documentaries.",
    "灵小柔": "🇨🇳 Chinese · Female (Young Adult) · Clear, sweet and articulate; polite, patient delivery for IVR & assistants.",
    "灵楷": "🇨🇳 Chinese · Male (Young Adult) · Warm and resonant; slow, soothing late-night radio, documentary & prose narration.",
    "灵小安": "🇨🇳 Chinese · Female (Teen) · Sweet, crisp and energetic; cute anime-style guide & AI assistant.",
    "灵语汐": "🇨🇳 Chinese · Female (Teen) · Clear, bright and youthful; sunny, energetic entertainment & assistant voice.",
    "灵小萌": "🇨🇳 Chinese · Female (Young Adult) · Bright, crisp and brisk; dependable audiobook & soothing radio host.",
    "灵小雅": "🇨🇳 Chinese · Female (Young Adult) · Sweet, bright and youthful; newsreader & audiobook presenter.",
    "花小呗": "🇨🇳 Chinese · Female (Teen/Child) · Crisp and sweet with a lively childlike character delivery.",
    "灵韵": "🇨🇳 Chinese · Female (Young Adult) · Bright and crisp with brisk, dependable customer-service delivery.",
    "妩媚妲己": "🇨🇳 Chinese · Female (Young Adult) · Sweet and alluring character voice with light, rising intonation.",
    "灵婉": "🇨🇳 Chinese · Female (Teen) · Youthful, intimate and soothing delivery for late-night radio & audiobooks.",
    "灵朵儿": "🇨🇳 Chinese · Female (Young Adult) · Warm, thoughtful mentor & cultural-documentary narrator.",
    "灵小锋": "🇨🇳 Chinese · Male (Young Adult) · Warm and resonant with natural confidence for daily & dramatic dialogue.",
    "灵小川": "🇨🇳 Chinese · Male (Young Adult) · Warm, mellow radio host & expressive inner-monologue narrator.",
    "悟空": "🇨🇳 Chinese · Male · Highly expressive, theatrical oral storyteller for animation & picture books.",
}


def describe_builtin_voice(voice_id: str) -> str:
    if not voice_id:
        return ""
    if voice_id in VOICE_PERSONA_CATALOG:
        return f"**Selected Voice (`{voice_id}`):** {VOICE_PERSONA_CATALOG[voice_id]}"
    if "_" in voice_id and not voice_id.startswith(("US_", "UK_")):
        base, emo = voice_id.split("_", 1)
        return f"**Selected Voice (`{voice_id}`):** 🇨🇳 Chinese Emotion Variant · Base speaker `{base}` with explicit `{emo}` emotional expression."
    return f"**Selected Voice (`{voice_id}`):** 🇨🇳 Chinese Built-in Premium IP Voice."


SUPPORTED_LANGUAGES = ["Chinese", "English", "Japanese", "Korean", "German", "French", "Spanish"]
SUPPORTED_EMOTIONS = [
    "None",
    "高兴 (happy)",
    "悲伤 (sad)",
    "愤怒 (angry)",
    "恐惧 (fearful)",
    "厌恶 (disgusted)",
    "惊讶 (surprised)",
    "中性 (neutral)",
]
SUPPORTED_DIALECTS = [
    "None",
    "cantonese (广粤话 / Cantonese)",
    "chuanyu dialect (川渝话 / Sichuanese)",
]
SPEED_LEVELS = ["慢速 (Slow)", "中速 (Normal)", "快速 (Fast)"]
PITCH_VOLUME_LEVELS = ["低 (Low)", "中 (Medium)", "高 (High)"]
PARALINGUISTIC_TAGS = list(tts_tasks.PARALINGUISTIC_TAGS)

EXAMPLE_LINGGUANG_WAV = os.path.join(REPO_DIR, "data", "voice_presets", "voice_audio", "lingguang.wav")
EXAMPLE_LINGGUANG_TXT = "诶，你今天有空吗？我们能不能聊聊这件事啊？"
EXAMPLE_DB30_WAV = os.path.join(REPO_DIR, "data", "voice_presets", "voice_audio", "db30.wav")
EXAMPLE_DB30_TXT = "感谢你的认可。"
EXAMPLE_EMOTION_WAV = os.path.join(REPO_DIR, "data", "wavs", "emotion_prompt.wav")
EXAMPLE_YUE_WAV = os.path.join(REPO_DIR, "data", "wavs", "yue_prompt.wav")

# Lazy CPU Whisper ASR pipeline for 1-click reference audio transcription (0 MB GPU VRAM)
_ASR_PIPELINE = None


def transcribe_reference_audio_cpu(audio_path: str) -> str:
    global _ASR_PIPELINE
    if not audio_path or not os.path.exists(audio_path):
        raise ValueError("Please upload or select a reference audio clip first.")
    if _ASR_PIPELINE is None:
        from transformers import pipeline as hf_pipeline
        print("Loading lightweight openai/whisper-base on CPU for reference transcription (0 GPU VRAM)...")
        _ASR_PIPELINE = hf_pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-base",
            device="cpu",
        )
    data, sr = sf.read(audio_path, dtype="float32", always_2d=True)
    mono = data.mean(axis=1)
    if sr != 16000:
        wf = torch.from_numpy(mono).unsqueeze(0)
        mono = torchaudio.transforms.Resample(sr, 16000)(wf).squeeze(0).numpy()
    result = _ASR_PIPELINE({"array": mono, "sampling_rate": 16000})
    text = (result.get("text") or "").strip() if isinstance(result, dict) else str(result).strip()
    return text


def build_structured_voice_sheet(
    gender: str,
    age: str,
    pitch_desc: str,
    speed_desc: str,
    volume_desc: str,
    timbre_desc: str,
    emotion_desc: str,
    intonation_desc: str,
    personality_desc: str,
    accent_desc: str = "标准普通话，自然叙事风格",
) -> str:
    '''Build the exact 12-field structured Chinese voice-design caption from inclusionAI's cookbook.'''
    lines = [
        f"性别: {gender}.",
        f"音高: {pitch_desc}.",
        f"语速: {speed_desc}.",
        f"音量: {volume_desc}.",
        f"年龄: {age}.",
        "清晰度: 吐字清晰，发音圆润.",
        "流畅度: 表达流利，一气呵成.",
        f"口音: {accent_desc}.",
        f"音色质感: {timbre_desc}.",
        f"情绪: {emotion_desc}.",
        f"语调: {intonation_desc}.",
        f"性格: {personality_desc}.",
    ]
    return "\n\n".join(lines)


def _infer_en_or_zh_language_id(text: str, voice_id: str = "") -> Optional[str]:
    if voice_id and voice_id.startswith(("US_", "UK_")):
        return "en"
    if not text:
        return None
    cjk = sum(1 for ch in text if "\u4e00" <= ch <= "\u9fff")
    ascii_alpha = sum(1 for ch in text if ("A" <= ch <= "Z") or ("a" <= ch <= "z"))
    if ascii_alpha > 0 and cjk == 0:
        return "en"
    if cjk > 0:
        return "zh"
    return None


def seed_everything(seed: int) -> int:
    seed = int(seed)
    if seed < 0:
        seed = random.randint(0, 2**31 - 1)
    random.seed(seed)
    np.random.seed(seed % (2**32))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return seed


def normalize_peak(audio_np: np.ndarray, target_peak: float = 0.92) -> np.ndarray:
    arr = np.asarray(audio_np, dtype=np.float32).reshape(-1)
    peak = float(np.max(np.abs(arr))) if arr.size else 0.0
    if peak > 1e-4:
        arr = arr * (target_peak / peak)
    return np.clip(arr, -1.0, 1.0)


def save_output_wav(waveform_np: np.ndarray, sample_rate: int, prefix: str = "ming3") -> str:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_prefix = re.sub(r"[^A-Za-z0-9_-]+", "_", prefix).strip("_") or "ming3"
    out_path = os.path.join(OUTPUT_DIR, f"{safe_prefix}_{ts}.wav")
    audio_f32 = np.clip(np.asarray(waveform_np, dtype=np.float32).reshape(-1), -1.0, 1.0)
    sf.write(out_path, audio_f32, sample_rate, subtype="PCM_16")
    return out_path


def list_drive_voices() -> List[str]:
    os.makedirs(VOICES_LIB_DIR, exist_ok=True)
    presets = []
    if os.path.exists(EXAMPLE_LINGGUANG_WAV):
        presets.append("[Built-in Example] lingguang.wav")
    if os.path.exists(EXAMPLE_DB30_WAV):
        presets.append("[Built-in Example] db30.wav")
    if os.path.exists(EXAMPLE_EMOTION_WAV):
        presets.append("[Built-in Example] emotion_prompt.wav")
    if os.path.exists(EXAMPLE_YUE_WAV):
        presets.append("[Built-in Example] yue_prompt.wav")
    for ext in ("*.wav", "*.mp3", "*.flac", "*.ogg"):
        for fpath in sorted(glob.glob(os.path.join(VOICES_LIB_DIR, ext))):
            presets.append(os.path.basename(fpath))
    return presets


def resolve_drive_voice(selection: str) -> Tuple[Optional[str], str]:
    if not selection:
        return None, ""
    if selection == "[Built-in Example] lingguang.wav":
        return EXAMPLE_LINGGUANG_WAV, EXAMPLE_LINGGUANG_TXT
    if selection == "[Built-in Example] db30.wav":
        return EXAMPLE_DB30_WAV, EXAMPLE_DB30_TXT
    if selection == "[Built-in Example] emotion_prompt.wav":
        return EXAMPLE_EMOTION_WAV, ""
    if selection == "[Built-in Example] yue_prompt.wav":
        return EXAMPLE_YUE_WAV, ""
    candidate = os.path.join(VOICES_LIB_DIR, selection)
    txt_candidate = os.path.splitext(candidate)[0] + ".txt"
    ref_txt = ""
    if os.path.exists(txt_candidate):
        with open(txt_candidate, "r", encoding="utf-8") as tf:
            ref_txt = tf.read().strip()
    return (candidate if os.path.exists(candidate) else None), ref_txt


def save_voice_to_drive(audio_path: str, transcript: str, voice_name: str) -> str:
    if not audio_path or not os.path.exists(audio_path):
        raise ValueError("Generate, upload, or record an audio clip first.")
    clean_name = re.sub(r"[^A-Za-z0-9_-]+", "_", (voice_name or "").strip()).strip("_")
    if not clean_name:
        clean_name = f"voice_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(VOICES_LIB_DIR, exist_ok=True)
    dst_wav = os.path.join(VOICES_LIB_DIR, f"{clean_name}.wav")
    data, sr = sf.read(audio_path, dtype="float32", always_2d=True)
    sf.write(dst_wav, data, sr, subtype="PCM_16")
    if transcript and transcript.strip():
        with open(os.path.join(VOICES_LIB_DIR, f"{clean_name}.txt"), "w", encoding="utf-8") as tf:
            tf.write(transcript.strip())
    return f"Saved '{clean_name}.wav' (+ transcript) to Drive Voice Library ({VOICES_LIB_DIR})"


class Ming3StudioEngine:
    '''Unified manager for Ming-3.0-tts-0.5B (Dense) and Ming-3.0-tts-tiny (MoE).'''

    VARIANTS = {
        "Ming-3.0-tts-0.5B (Dense 0.5B · Fast · ~4GB VRAM)": "0.5B",
        "Ming-3.0-tts-tiny (MoE 7.9B/1.3B · Native Podcast · ~19GB VRAM)": "tiny",
    }

    def __init__(self, device: str = "cuda:0"):
        self.device = device
        self.current_variant: Optional[str] = None
        self.dense_model: Optional[MingTTS] = None
        self.tiny_model = None
        self.sample_rate: int = 44100

    def unload(self):
        self.dense_model = None
        self.tiny_model = None
        self.current_variant = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def load_variant(self, variant_label: str) -> str:
        variant_key = self.VARIANTS.get(variant_label, variant_label)
        if variant_key not in ("0.5B", "tiny"):
            variant_key = "0.5B"
        if self.current_variant == variant_key and (self.dense_model is not None or self.tiny_model is not None):
            return variant_key

        self.unload()
        from huggingface_hub import snapshot_download

        if variant_key == "0.5B":
            local_dir = MODEL_05B_DIR if os.path.exists(os.path.join(MODEL_05B_DIR, "config.json")) else snapshot_download(
                repo_id=MODEL_05B_REPO,
                local_dir=MODEL_05B_DIR,
                ignore_patterns=["*.pdf", "figures/*"],
            )
            print(f"Loading Ming-3.0-tts-0.5B (Dense) from {local_dir} ...")
            self.dense_model = MingTTS(
                local_dir,
                device=self.device,
                text_normalization=HAS_PYNINI,
                cache_prompts=False,
            )
            if HAS_PYNINI:
                try:
                    _ = self.dense_model.normalizer
                except Exception as norm_err:
                    print(f"Normalizer init fallback ({norm_err}); disabling FST normalizer.")
                    self.dense_model.text_normalization = False
            self.sample_rate = int(self.dense_model.sample_rate)
            self.current_variant = "0.5B"
        else:
            vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
            if vram_gb < 18.0:
                raise RuntimeError(
                    f"Ming-3.0-tts-tiny (7.9B MoE, 16.8 GB weights) requires ~19 GB VRAM, "
                    f"but current GPU has {vram_gb:.1f} GB. Please switch to 'Ming-3.0-tts-0.5B (Dense)' or use an L4/A100 GPU."
                )
            local_dir = MODEL_TINY_DIR if os.path.exists(os.path.join(MODEL_TINY_DIR, "config.json")) else snapshot_download(
                repo_id=MODEL_TINY_REPO,
                local_dir=MODEL_TINY_DIR,
                ignore_patterns=["*.pdf", "figures/*"],
            )
            print(f"Loading Ming-3.0-tts-tiny (MoE 7.9B) from {local_dir} ...")
            from ming_tiny import inference as ming_tiny_inference
            from ming_tiny.api import MingTinyTTS

            def _tiny_safe_waveform(path: str, target_sample_rate: int = 16000):
                data, sr = sf.read(path, dtype="float32", always_2d=True)
                wf = torch.from_numpy(data.T.copy()).float().mean(dim=0, keepdim=True)
                if sr != target_sample_rate:
                    wf = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sample_rate)(wf)
                return wf

            ming_tiny_inference.load_prompt_waveform = _tiny_safe_waveform
            self.tiny_model = MingTinyTTS.from_pretrained(local_dir, device=self.device)
            self.sample_rate = int(self.tiny_model.sample_rate)
            self.current_variant = "tiny"

        return self.current_variant

    def synthesize_zero_shot(
        self,
        variant_label: str,
        text: str,
        ref_audio: str,
        ref_text: str = "",
        language: str = "Chinese",
        seed: int = 1895,
        cfg: float = 2.0,
        sigma: float = 0.25,
        temperature: float = 0.0,
        max_length: int = 50,
        trim_silence: bool = True,
    ) -> Tuple[Tuple[int, np.ndarray], str, str]:
        text = (text or "").strip()
        if not text:
            raise ValueError("Please enter text to synthesize.")
        if not ref_audio or not os.path.exists(ref_audio):
            raise ValueError("Please provide a valid reference audio clip (3–10s).")
        ref_text_clean = (ref_text or "").strip() or None

        vkey = self.load_variant(variant_label)
        used_seed = seed_everything(seed)
        t0 = time.perf_counter()

        if vkey == "0.5B":
            task = "TTS" if language in ("Chinese", "English") else "MULTILINGUAL"
            wav = self.dense_model.synthesize(
                task=task,
                text=text,
                language_id=language,
                prompt_wav_path=ref_audio,
                prompt_text=ref_text_clean,
                use_spk_emb=True,
                cfg=float(cfg),
                sigma=float(sigma),
                temperature=float(temperature),
                max_length=int(max_length),
                trim_silence=bool(trim_silence),
            )
            audio_np = wav.squeeze(0).detach().float().cpu().clamp(-1.0, 1.0).numpy()
        else:
            if not ref_text_clean:
                raise ValueError(
                    "Ming-3.0-tts-tiny voice cloning requires the reference transcript. "
                    "Click '🪄 Auto-Transcribe Reference Audio (CPU Whisper)' to fill it automatically!"
                )
            if language in ("Chinese", "English"):
                task = "paralinguistic" if "[" in text else "seed"
            else:
                task = "multilingual"
            out = self.tiny_model.generate(
                task=task,
                text=text,
                prompt_wav=ref_audio,
                prompt_text=ref_text_clean,
                target_language=language,
                seed=used_seed,
                cfg_scale=float(cfg),
                sigma=float(sigma),
                temperature=float(temperature),
            )
            audio_np = out.waveform.detach().float().cpu().clamp(-1.0, 1.0).numpy().reshape(-1)

        dt = time.perf_counter() - t0
        dur = len(audio_np) / self.sample_rate
        out_path = save_output_wav(audio_np, self.sample_rate, f"ming3_{vkey}_clone_{language.lower()}")
        status = f"✅ Generated {dur:.2f}s @ {self.sample_rate} Hz in {dt:.2f}s (model={vkey}, task={task}, seed={used_seed})\nSaved to: {out_path}"
        return (self.sample_rate, (audio_np * 32767).astype(np.int16)), out_path, status

    def synthesize_controlled(
        self,
        variant_label: str,
        control_mode: str,
        text: str,
        ref_audio: str,
        ref_text: str = "",
        emotion: str = "高兴 (happy)",
        dialect: str = "cantonese (广粤话 / Cantonese)",
        speed: str = "中速 (Normal)",
        pitch: str = "中 (Medium)",
        volume: str = "中 (Medium)",
        seed: int = 1895,
        cfg: float = 2.0,
        sigma: float = 0.25,
        temperature: float = 0.0,
        max_length: int = 50,
        trim_silence: bool = True,
    ) -> Tuple[Tuple[int, np.ndarray], str, str]:
        text = (text or "").strip()
        if not text:
            raise ValueError("Please enter text to synthesize.")
        if not ref_audio or not os.path.exists(ref_audio):
            raise ValueError("Please provide a reference audio clip for speaker timbre.")
        ref_text_clean = (ref_text or "").strip() or None

        vkey = self.load_variant(variant_label)
        used_seed = seed_everything(seed)
        t0 = time.perf_counter()

        emo_val = (emotion or "高兴").split(" ")[0]
        if emo_val == "None":
            emo_val = "高兴"
        dia_raw = (dialect or "cantonese").split(" (")[0].strip()
        if dia_raw == "None":
            dia_raw = "cantonese"
        spd_val = (speed or "中速").split(" ")[0]
        pit_val = (pitch or "中").split(" ")[0]
        vol_val = (volume or "中").split(" ")[0]

        if vkey == "0.5B":
            if control_mode == "Emotion Control":
                # Emotion control uses speaker embedding only so the reference clip's neutral prosody doesn't override the target emotion
                wav = self.dense_model.synthesize(
                    task="EMOTION",
                    text=text,
                    controls={"emotion": emo_val},
                    prompt_wav_path=ref_audio,
                    use_spk_emb=True,
                    cfg=float(cfg),
                    sigma=float(sigma),
                    temperature=float(temperature),
                    max_length=int(max_length),
                    trim_silence=bool(trim_silence),
                )
            elif control_mode == "Dialect Control (Cantonese / Sichuanese)":
                wav = self.dense_model.synthesize(
                    task="DIALECT",
                    text=text,
                    controls={"dialect": dia_raw},
                    prompt_wav_path=ref_audio,
                    prompt_text=ref_text_clean,
                    use_spk_emb=True,
                    cfg=float(cfg),
                    sigma=float(sigma),
                    temperature=float(temperature),
                    max_length=int(max_length),
                    trim_silence=bool(trim_silence),
                )
            else:
                wav = self.dense_model.synthesize(
                    task="BASIC",
                    text=text,
                    controls={"speed": spd_val, "pitch": pit_val, "volume": vol_val},
                    prompt_wav_path=ref_audio,
                    prompt_text=ref_text_clean,
                    use_spk_emb=True,
                    cfg=float(cfg),
                    sigma=float(sigma),
                    temperature=float(temperature),
                    max_length=int(max_length),
                    trim_silence=bool(trim_silence),
                )
            audio_np = wav.squeeze(0).detach().float().cpu().clamp(-1.0, 1.0).numpy()
        else:
            if control_mode == "Emotion Control":
                out = self.tiny_model.generate(
                    task="emotion",
                    text=text,
                    prompt_wav=ref_audio,
                    emotion=emo_val,
                    seed=used_seed,
                    cfg_scale=float(cfg),
                    sigma=float(sigma),
                    temperature=float(temperature),
                )
            elif control_mode == "Dialect Control (Cantonese / Sichuanese)":
                dia_zh = "广粤话" if "canton" in dia_raw.lower() else "川渝话"
                out = self.tiny_model.generate(
                    task="dialect",
                    text=text,
                    prompt_wav=ref_audio,
                    dialect=dia_zh,
                    seed=used_seed,
                    cfg_scale=float(cfg),
                    sigma=float(sigma),
                    temperature=float(temperature),
                )
            else:
                out = self.tiny_model.generate(
                    task="basic",
                    text=text,
                    prompt_wav=ref_audio,
                    speed=spd_val,
                    pitch=pit_val,
                    volume=vol_val,
                    seed=used_seed,
                    cfg_scale=float(cfg),
                    sigma=float(sigma),
                    temperature=float(temperature),
                )
            audio_np = out.waveform.detach().float().cpu().clamp(-1.0, 1.0).numpy().reshape(-1)

        dt = time.perf_counter() - t0
        dur = len(audio_np) / self.sample_rate
        out_path = save_output_wav(audio_np, self.sample_rate, f"ming3_{vkey}_control")
        status = f"✅ Generated {dur:.2f}s @ {self.sample_rate} Hz in {dt:.2f}s (model={vkey}, mode={control_mode}, seed={used_seed})\nSaved to: {out_path}"
        return (self.sample_rate, (audio_np * 32767).astype(np.int16)), out_path, status

    def synthesize_design_or_freeform(
        self,
        variant_label: str,
        text: str,
        description: str,
        mode: str = "Voice Design",
        optional_ref_audio: Optional[str] = None,
        seed: int = 1895,
        cfg: float = 2.0,
        sigma: float = 0.25,
        temperature: float = 0.0,
        max_length: int = 80,
        trim_silence: bool = True,
    ) -> Tuple[Tuple[int, np.ndarray], str, str]:
        text = (text or "").strip()
        description = (description or "").strip()
        if not text:
            raise ValueError("Please enter text to synthesize.")
        if not description:
            raise ValueError("Please enter a voice description or delivery instruction.")
        has_ref = bool(optional_ref_audio and os.path.exists(optional_ref_audio) and mode == "Freeform")

        vkey = self.load_variant(variant_label)
        used_seed = seed_everything(seed)
        t0 = time.perf_counter()

        if vkey == "0.5B":
            lang_hint = _infer_en_or_zh_language_id(text)
            if mode == "Freeform":
                wav = self.dense_model.synthesize(
                    task="FREEFORM",
                    text=text,
                    language_id=lang_hint,
                    instruction=description,
                    prompt_wav_path=optional_ref_audio if has_ref else None,
                    use_spk_emb=has_ref,
                    use_zero_spk_emb=not has_ref,
                    cfg=float(cfg),
                    sigma=float(sigma),
                    temperature=float(temperature),
                    max_length=int(max_length),
                    trim_silence=bool(trim_silence),
                )
            else:
                wav = self.dense_model.synthesize(
                    task="VOICE_DESIGN",
                    text=text,
                    language_id=lang_hint,
                    controls={"style": description},
                    use_zero_spk_emb=True,
                    cfg=float(cfg),
                    sigma=float(sigma),
                    temperature=float(temperature),
                    max_length=int(max_length),
                    trim_silence=bool(trim_silence),
                )
            audio_np = wav.squeeze(0).detach().float().cpu().clamp(-1.0, 1.0).numpy()
        else:
            if mode == "Freeform":
                out = self.tiny_model.generate(
                    task="freeform",
                    text=text,
                    instruction=description,
                    prompt_wav=optional_ref_audio if has_ref else None,
                    seed=used_seed,
                    cfg_scale=float(cfg),
                    sigma=float(sigma),
                    temperature=float(temperature),
                )
            else:
                out = self.tiny_model.generate(
                    task="voice_design",
                    text=text,
                    style=description,
                    seed=used_seed,
                    cfg_scale=float(cfg),
                    sigma=float(sigma),
                    temperature=float(temperature),
                )
            audio_np = out.waveform.detach().float().cpu().clamp(-1.0, 1.0).numpy().reshape(-1)

        dt = time.perf_counter() - t0
        dur = len(audio_np) / self.sample_rate
        out_path = save_output_wav(audio_np, self.sample_rate, f"ming3_{vkey}_{mode.lower().replace(' ', '_')}")
        status = f"✅ Generated {dur:.2f}s @ {self.sample_rate} Hz in {dt:.2f}s (model={vkey}, mode={mode}, seed={used_seed})\nSaved to: {out_path}"
        return (self.sample_rate, (audio_np * 32767).astype(np.int16)), out_path, status

    def synthesize_builtin_voice(
        self,
        variant_label: str,
        text: str,
        voice_id: str = "US_James",
        seed: int = 1895,
        cfg: float = 2.0,
        sigma: float = 0.25,
        temperature: float = 0.0,
        max_length: int = 50,
        trim_silence: bool = True,
    ) -> Tuple[Tuple[int, np.ndarray], str, str]:
        text = (text or "").strip()
        voice_id = (voice_id or "").strip()
        if not text:
            raise ValueError("Please enter text to synthesize.")
        if not voice_id:
            raise ValueError("Please select a built-in voice ID.")

        vkey = self.load_variant(variant_label)
        used_seed = seed_everything(seed)
        t0 = time.perf_counter()

        if vkey == "0.5B":
            lang_hint = _infer_en_or_zh_language_id(text, voice_id=voice_id)
            wav = self.dense_model.synthesize(
                task="IP",
                text=text,
                language_id=lang_hint,
                controls={"ip": voice_id},
                use_zero_spk_emb=True,
                cfg=float(cfg),
                sigma=float(sigma),
                temperature=float(temperature),
                max_length=int(max_length),
                trim_silence=bool(trim_silence),
            )
            audio_np = wav.squeeze(0).detach().float().cpu().clamp(-1.0, 1.0).numpy()
        else:
            out = self.tiny_model.generate(
                task="ip",
                text=text,
                ip=voice_id,
                seed=used_seed,
                cfg_scale=float(cfg),
                sigma=float(sigma),
                temperature=float(temperature),
            )
            audio_np = out.waveform.detach().float().cpu().clamp(-1.0, 1.0).numpy().reshape(-1)

        dt = time.perf_counter() - t0
        dur = len(audio_np) / self.sample_rate
        out_path = save_output_wav(audio_np, self.sample_rate, f"ming3_{vkey}_ip_{voice_id}")
        status = f"✅ Generated {dur:.2f}s @ {self.sample_rate} Hz in {dt:.2f}s (model={vkey}, voice={voice_id}, seed={used_seed})\nSaved to: {out_path}"
        return (self.sample_rate, (audio_np * 32767).astype(np.int16)), out_path, status

    def synthesize_podcast(
        self,
        variant_label: str,
        script: str,
        spk1_audio: str,
        spk1_text: str,
        spk2_audio: str,
        spk2_text: str,
        language: str = "Chinese",
        pause_sec: float = 0.25,
        normalize_audio: bool = True,
        seed: int = 1235,
        cfg: float = 2.0,
        sigma: float = 0.25,
    ) -> Tuple[Tuple[int, np.ndarray], str, str]:
        script = (script or "").strip()
        if not script:
            raise ValueError("Please enter a dialogue script with lines starting with 'speaker_1:' or 'speaker_2:'.")
        if not spk1_audio or not os.path.exists(spk1_audio):
            raise ValueError("Please provide Speaker 1 reference audio.")
        if not spk2_audio or not os.path.exists(spk2_audio):
            raise ValueError("Please provide Speaker 2 reference audio.")

        vkey = self.load_variant(variant_label)
        used_seed = seed_everything(seed)
        t0 = time.perf_counter()

        if vkey == "tiny":
            if not (spk1_text or "").strip() or not (spk2_text or "").strip():
                raise ValueError(
                    "Ming-3.0-tts-tiny native podcast mode requires transcripts for both Speaker 1 and Speaker 2. "
                    "Use the '🪄 Auto-Transcribe' buttons if needed!"
                )
            out = self.tiny_model.generate(
                task="podcast",
                text=script,
                prompt_wav=[spk1_audio, spk2_audio],
                prompt_text=[spk1_text.strip(), spk2_text.strip()],
                seed=used_seed,
                cfg_scale=float(cfg),
                sigma=float(sigma),
            )
            audio_np = out.waveform.detach().float().cpu().clamp(-1.0, 1.0).numpy().reshape(-1)
            mode_desc = "tiny-native-podcast"
        else:
            # Turn-by-turn stitched multi-speaker synthesis for Ming-3.0-tts-0.5B (works on 16GB T4!)
            turns = []
            for raw_line in script.splitlines():
                line = raw_line.strip()
                if not line:
                    continue
                m = re.match(r"^(?:speaker[_\s]*([12])|\[S([12])\])\s*[:：]\s*(.+)$", line, flags=re.IGNORECASE)
                if m:
                    spk_idx = int(m.group(1) or m.group(2))
                    utterance = m.group(3).strip()
                else:
                    spk_idx = 1 if len(turns) % 2 == 0 else 2
                    utterance = line
                if utterance:
                    turns.append((spk_idx, utterance))

            if not turns:
                raise ValueError("No valid dialogue lines found in script.")

            task = "TTS" if language in ("Chinese", "English") else "MULTILINGUAL"
            segments = []
            pause_samples = int(max(0.0, float(pause_sec)) * self.sample_rate)
            silence_gap = np.zeros(pause_samples, dtype=np.float32)

            for idx, (spk_idx, utterance) in enumerate(turns):
                ref_w = spk1_audio if spk_idx == 1 else spk2_audio
                ref_t = (spk1_text if spk_idx == 1 else spk2_text or "").strip() or None
                wav = self.dense_model.synthesize(
                    task=task,
                    text=utterance,
                    language_id=language,
                    prompt_wav_path=ref_w,
                    prompt_text=ref_t,
                    use_spk_emb=True,
                    cfg=float(cfg),
                    sigma=float(sigma),
                )
                seg_np = wav.squeeze(0).detach().float().cpu().clamp(-1.0, 1.0).numpy()
                if normalize_audio:
                    seg_np = normalize_peak(seg_np, target_peak=0.90)
                segments.append(seg_np)
                if idx < len(turns) - 1 and pause_samples > 0:
                    segments.append(silence_gap)

            audio_np = np.concatenate(segments, axis=0)
            mode_desc = f"0.5B-stitched ({len(turns)} turns)"

        if normalize_audio:
            audio_np = normalize_peak(audio_np, target_peak=0.92)

        dt = time.perf_counter() - t0
        dur = len(audio_np) / self.sample_rate
        out_path = save_output_wav(audio_np, self.sample_rate, f"ming3_{vkey}_podcast")
        status = f"✅ Generated {dur:.2f}s podcast @ {self.sample_rate} Hz in {dt:.2f}s (mode={mode_desc}, seed={used_seed})\nSaved to: {out_path}"
        return (self.sample_rate, (audio_np * 32767).astype(np.int16)), out_path, status


ENGINE = Ming3StudioEngine(device="cuda:0")
# Pre-load the lightweight 0.5B Dense model by default so the UI is ready immediately
try:
    ENGINE.load_variant("0.5B")
    print(f"Ming3StudioEngine ready! Default active model: {ENGINE.current_variant} @ {ENGINE.sample_rate} Hz")
except Exception as e:
    print(f"Deferred model loading until first UI synthesis call: {e}")


In [ ]:
#@title Step 4: Launch Ming 3.0 TTS 5-Tab Gradio Studio
import os
import torch
import gradio as gr
from IPython.display import clear_output

MODEL_CHOICES = list(Ming3StudioEngine.VARIANTS.keys())
DEFAULT_MODEL_CHOICE = MODEL_CHOICES[0]
VOICE_CATEGORY_CHOICES = ["All Voices (482)", "English (5)", "Chinese Base (48)", "Chinese Emotion Variants (429)"]


def get_system_welcome() -> str:
    gpu_info = "CPU"
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        used = torch.cuda.memory_allocated(0) / (1024 ** 3)
        gpu_info = f"{name} ({used:.1f} / {total:.1f} GB VRAM allocated)"
    tn_state = "Active (pynini FST)" if HAS_PYNINI else "Fallback (pynini disabled)"
    return (
        f"**GPU Runtime:** `{gpu_info}` · **Sample Rate:** `44.1 kHz` · "
        f"**Text Normalizer:** `{tn_state}` · **Drive Output:** `{OUTPUT_DIR}`"
    )


def filter_voice_choices(category: str):
    if category == "English (5)":
        choices = VOICE_GROUPS.get("English", ALL_VOICES[:5])
    elif category == "Chinese Base (48)":
        choices = VOICE_GROUPS.get("Chinese", ALL_VOICES)
    elif category == "Chinese Emotion Variants (429)":
        choices = VOICE_GROUPS.get("Chinese (emotion variants)", ALL_VOICES)
    else:
        choices = ALL_VOICES
    default_val = choices[0] if choices else "US_James"
    return gr.update(choices=choices, value=default_val), describe_builtin_voice(default_val)


def append_para_tag(current_text: str, tag: str) -> str:
    current_text = current_text or ""
    if current_text and not current_text.endswith(" "):
        return f"{current_text} {tag} "
    return f"{current_text}{tag} "


def ui_load_drive_voice(selection: str):
    wav_path, ref_txt = resolve_drive_voice(selection)
    return wav_path, ref_txt


def ui_save_drive_voice(audio_path: str, transcript: str, voice_name: str):
    try:
        msg = save_voice_to_drive(audio_path, transcript, voice_name)
        updated = list_drive_voices()
        return gr.update(choices=updated), gr.update(choices=updated), gr.update(choices=updated), msg
    except Exception as e:
        raise gr.Error(str(e))


def ui_transcribe_clip(audio_path: str) -> str:
    try:
        return transcribe_reference_audio_cpu(audio_path)
    except Exception as e:
        raise gr.Error(f"Auto-transcription failed: {e}")


def ui_build_sheet(gender, age, pitch_d, speed_d, vol_d, timbre_d, emo_d, inton_d, pers_d):
    return build_structured_voice_sheet(
        gender=gender,
        age=age,
        pitch_desc=pitch_d,
        speed_desc=speed_d,
        volume_desc=vol_d,
        timbre_desc=timbre_d,
        emotion_desc=emo_d,
        intonation_desc=inton_d,
        personality_desc=pers_d,
    ), "Voice Design"


def ui_clone_voice(model_choice, text, ref_audio, ref_text, language, seed, cfg, sigma, temp, max_len, trim_sil):
    try:
        return ENGINE.synthesize_zero_shot(
            variant_label=model_choice,
            text=text,
            ref_audio=ref_audio,
            ref_text=ref_text,
            language=language,
            seed=seed,
            cfg=cfg,
            sigma=sigma,
            temperature=temp,
            max_length=max_len,
            trim_silence=trim_sil,
        )
    except Exception as e:
        raise gr.Error(str(e))


def ui_controlled_voice(model_choice, control_mode, text, ref_audio, ref_text, emotion, dialect, speed, pitch, volume, seed, cfg, sigma, temp, max_len, trim_sil):
    try:
        return ENGINE.synthesize_controlled(
            variant_label=model_choice,
            control_mode=control_mode,
            text=text,
            ref_audio=ref_audio,
            ref_text=ref_text,
            emotion=emotion,
            dialect=dialect,
            speed=speed,
            pitch=pitch,
            volume=volume,
            seed=seed,
            cfg=cfg,
            sigma=sigma,
            temperature=temp,
            max_length=max_len,
            trim_silence=trim_sil,
        )
    except Exception as e:
        raise gr.Error(str(e))


def ui_design_voice(model_choice, text, description, mode, opt_ref_audio, seed, cfg, sigma, temp, max_len, trim_sil):
    try:
        return ENGINE.synthesize_design_or_freeform(
            variant_label=model_choice,
            text=text,
            description=description,
            mode=mode,
            optional_ref_audio=opt_ref_audio,
            seed=seed,
            cfg=cfg,
            sigma=sigma,
            temperature=temp,
            max_length=max_len,
            trim_silence=trim_sil,
        )
    except Exception as e:
        raise gr.Error(str(e))


def ui_builtin_voice(model_choice, text, voice_id, seed, cfg, sigma, temp, max_len, trim_sil):
    try:
        return ENGINE.synthesize_builtin_voice(
            variant_label=model_choice,
            text=text,
            voice_id=voice_id,
            seed=seed,
            cfg=cfg,
            sigma=sigma,
            temperature=temp,
            max_length=max_len,
            trim_silence=trim_sil,
        )
    except Exception as e:
        raise gr.Error(str(e))


def ui_podcast(model_choice, script, spk1_audio, spk1_text, spk2_audio, spk2_text, language, pause_sec, norm_audio, seed, cfg, sigma):
    try:
        return ENGINE.synthesize_podcast(
            variant_label=model_choice,
            script=script,
            spk1_audio=spk1_audio,
            spk1_text=spk1_text,
            spk2_audio=spk2_audio,
            spk2_text=spk2_text,
            language=language,
            pause_sec=pause_sec,
            normalize_audio=norm_audio,
            seed=seed,
            cfg=cfg,
            sigma=sigma,
        )
    except Exception as e:
        raise gr.Error(str(e))


clear_output()

with gr.Blocks(title="Ming 3.0 TTS Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 🎙️ Ming 3.0 TTS Studio (`Ming-3.0-tts-0.5B` Dense & `Ming-3.0-tts-tiny` MoE)\n"
        "44.1 kHz multilingual speech synthesis with zero-shot voice cloning, 14 inline paralinguistic tags, "
        "voice design, free-form delivery instructions, 482 built-in voices, emotion/dialect/prosody control, and two-speaker podcasts."
    )
    welcome_md = gr.Markdown(value=get_system_welcome())

    with gr.Row():
        model_selector = gr.Dropdown(
            choices=MODEL_CHOICES,
            value=DEFAULT_MODEL_CHOICE,
            label="Active Ming 3.0 TTS Model Variant",
            info="0.5B Dense fits on any GPU (~4 GB VRAM, T4/L4/A100). Tiny MoE (7.9B total / 1.3B active) requires L4 24GB or A100 40GB (~19 GB VRAM).",
        )

    with gr.Tabs():
        # =====================================================================
        # TAB 1: Zero-Shot Voice Cloning & Paralinguistic Tags
        # =====================================================================
        with gr.Tab("🎙️ 1. Zero-Shot Voice Cloning & Tags"):
            with gr.Row():
                with gr.Column(scale=6):
                    c_text = gr.Textbox(
                        label="Text to Synthesize",
                        lines=4,
                        value="The garden will remain open until sunset, provided the weather stays clear.",
                        placeholder="Enter text in Chinese, English, Japanese, Korean, German, French, or Spanish. Click any tag button below to insert paralinguistic expressions!",
                        info="Supports inline paralinguistic tags like [Laughter], [Sigh], [Breathing], [Cough], [Uhm] in Chinese & English.",
                    )
                    gr.Markdown("**Quick-Insert Paralinguistic Tags (Click to append to text):**")
                    with gr.Row():
                        tag_btns_row1 = [gr.Button(t, size="sm") for t in PARALINGUISTIC_TAGS[:7]]
                    with gr.Row():
                        tag_btns_row2 = [gr.Button(t, size="sm") for t in PARALINGUISTIC_TAGS[7:]]
                    for b in tag_btns_row1 + tag_btns_row2:
                        b.click(fn=append_para_tag, inputs=[c_text, b], outputs=[c_text])

                    with gr.Row():
                        c_drive_voice = gr.Dropdown(
                            choices=list_drive_voices(),
                            value="[Built-in Example] lingguang.wav",
                            label="📂 Load Preset / Drive Voice Library",
                            info="Select a built-in reference clip or a voice saved in /content/drive/MyDrive/AEI_TTS_Cache/voices/.",
                        )
                        c_load_btn = gr.Button("Load Selected Voice", size="sm")

                    c_ref_audio = gr.Audio(
                        label="Reference Audio (3–10 seconds)",
                        type="filepath",
                        value=EXAMPLE_LINGGUANG_WAV if os.path.exists(EXAMPLE_LINGGUANG_WAV) else None,
                    )
                    with gr.Row():
                        c_ref_text = gr.Textbox(
                            label="Reference Transcript (Strongly Recommended)",
                            lines=2,
                            value=EXAMPLE_LINGGUANG_TXT,
                            info="Exact transcript of the reference clip enables 12.5 Hz AudioVAE latent prompt continuation for maximum speaker similarity.",
                        )
                        c_asr_btn = gr.Button("🪄 Auto-Transcribe (CPU Whisper)", size="sm")

                    c_lang = gr.Dropdown(
                        choices=SUPPORTED_LANGUAGES,
                        value="English",
                        label="Target Language",
                        info="Select the language of the synthesized text (Chinese, English, Japanese, Korean, German, French, Spanish).",
                    )
                    with gr.Accordion("💾 Save Reference Voice to Google Drive Library", open=False):
                        with gr.Row():
                            c_save_name = gr.Textbox(
                                label="Voice Preset Name",
                                placeholder="e.g. nikita_warm_studio",
                                info="Saves both the .wav and .txt transcript to Drive for reuse across all TTS notebooks.",
                            )
                            c_save_btn = gr.Button("Save to Drive Library")
                        c_save_status = gr.Textbox(label="Library Status", interactive=False, lines=1)

                    with gr.Accordion("⚙️ Advanced Sampling Settings", open=False):
                        with gr.Row():
                            c_seed = gr.Number(label="Random Seed (-1 = random)", value=1895, precision=0, info="Fixed integer seed for reproducible speech generation.")
                            c_cfg = gr.Slider(1.0, 5.0, value=2.0, step=0.1, label="CFG Scale", info="Classifier-free guidance scale of the flow-matching DiT head (default 2.0).")
                            c_sigma = gr.Slider(0.0, 1.0, value=0.25, step=0.05, label="Sigma (Noise)", info="Flow-matching sampling noise scale (default 0.25).")
                        with gr.Row():
                            c_temp = gr.Slider(0.0, 1.0, value=0.0, step=0.05, label="Temperature", info="Sampling temperature (default 0.0).")
                            c_maxlen = gr.Slider(16, 80, value=50, step=2, label="Max Fragment Length (0.5B)", info="Semantic clause splitting threshold on 0.5B.")
                            c_trim = gr.Checkbox(value=True, label="Trim Trailing Silence", info="Trim near-silent tail frames after utterance.")
                    c_btn = gr.Button("🎙️ Synthesize Cloned Speech", variant="primary")

                with gr.Column(scale=5):
                    c_audio_out = gr.Audio(label="Generated Speech (44.1 kHz)", type="numpy", interactive=False)
                    c_file_out = gr.File(label="Download .wav File (Saved to Drive AEI_TTS_Out)")
                    c_status = gr.Textbox(label="Generation Log", lines=3, interactive=False)

            c_load_btn.click(fn=ui_load_drive_voice, inputs=[c_drive_voice], outputs=[c_ref_audio, c_ref_text])
            c_asr_btn.click(fn=ui_transcribe_clip, inputs=[c_ref_audio], outputs=[c_ref_text])
            c_btn.click(
                fn=ui_clone_voice,
                inputs=[model_selector, c_text, c_ref_audio, c_ref_text, c_lang, c_seed, c_cfg, c_sigma, c_temp, c_maxlen, c_trim],
                outputs=[c_audio_out, c_file_out, c_status],
            )

        # =====================================================================
        # TAB 2: Emotion, Dialect & Prosody Attribute Control
        # =====================================================================
        with gr.Tab("🎭 2. Emotion, Dialect & Prosody Control"):
            with gr.Row():
                with gr.Column(scale=6):
                    ctrl_mode = gr.Radio(
                        choices=[
                            "Emotion Control",
                            "Dialect Control (Cantonese / Sichuanese)",
                            "Speed / Pitch / Volume (BASIC)",
                        ],
                        value="Emotion Control",
                        label="Structured Control Task Mode",
                        info="Ming 3.0 uses dedicated SFT task modes for Emotion, Dialect, or Speed/Pitch/Volume while cloning speaker timbre from the reference audio.",
                    )
                    ctrl_text = gr.Textbox(
                        label="Text to Synthesize",
                        lines=3,
                        value="所有测试都顺利通过了，我们终于可以把这个好消息告诉大家！",
                        info="Enter text matching your target emotion, dialect, or prosody style.",
                    )
                    ctrl_ref_audio = gr.Audio(
                        label="Reference Audio for Speaker Timbre",
                        type="filepath",
                        value=EXAMPLE_EMOTION_WAV if os.path.exists(EXAMPLE_EMOTION_WAV) else None,
                    )
                    ctrl_ref_text = gr.Textbox(
                        label="Optional Reference Transcript (Used on 0.5B for Dialect & BASIC modes)",
                        lines=2,
                        value="",
                        info="Optional transcript of the reference clip (boosts speaker similarity on 0.5B Dialect/BASIC tasks; automatically ignored for Emotion/Tiny).",
                    )
                    with gr.Row():
                        ctrl_emotion = gr.Dropdown(
                            choices=SUPPORTED_EMOTIONS[1:],
                            value="高兴 (happy)",
                            label="Target Emotion (for Emotion Control mode)",
                            info="Select one of the 7 trained emotional states.",
                        )
                        ctrl_dialect = gr.Dropdown(
                            choices=SUPPORTED_DIALECTS[1:],
                            value="cantonese (广粤话 / Cantonese)",
                            label="Target Dialect (for Dialect Control mode)",
                            info="Synthesize in Cantonese (广粤话) or Sichuanese (川渝话).",
                        )
                    with gr.Row():
                        ctrl_speed = gr.Dropdown(
                            choices=SPEED_LEVELS,
                            value="中速 (Normal)",
                            label="Speaking Speed (for BASIC mode)",
                            info="Control utterance speaking rate (慢速 / 中速 / 快速).",
                        )
                        ctrl_pitch = gr.Dropdown(
                            choices=PITCH_VOLUME_LEVELS,
                            value="中 (Medium)",
                            label="Pitch Level (for BASIC mode)",
                            info="Control fundamental frequency / pitch height (低 / 中 / 高).",
                        )
                        ctrl_volume = gr.Dropdown(
                            choices=PITCH_VOLUME_LEVELS,
                            value="中 (Medium)",
                            label="Volume / Energy Level (for BASIC mode)",
                            info="Control vocal projection energy (低 / 中 / 高).",
                        )
                    with gr.Accordion("⚙️ Advanced Sampling Settings", open=False):
                        with gr.Row():
                            ctrl_seed = gr.Number(label="Seed", value=1895, precision=0, info="Random seed (-1 = random).")
                            ctrl_cfg = gr.Slider(1.0, 5.0, value=2.0, step=0.1, label="CFG Scale", info="Flow-matching CFG scale.")
                            ctrl_sigma = gr.Slider(0.0, 1.0, value=0.25, step=0.05, label="Sigma", info="Sampling noise scale.")
                        with gr.Row():
                            ctrl_temp = gr.Slider(0.0, 1.0, value=0.0, step=0.05, label="Temperature", info="Sampling temperature.")
                            ctrl_maxlen = gr.Slider(16, 80, value=50, step=2, label="Max Fragment Length (0.5B)", info="Clause split length.")
                            ctrl_trim = gr.Checkbox(value=True, label="Trim Trailing Silence", info="Trim trailing silence.")
                    ctrl_btn = gr.Button("🎭 Generate Controlled Speech", variant="primary")

                with gr.Column(scale=5):
                    ctrl_audio_out = gr.Audio(label="Generated Speech (44.1 kHz)", type="numpy", interactive=False)
                    ctrl_file_out = gr.File(label="Download .wav File")
                    ctrl_status = gr.Textbox(label="Generation Log", lines=3, interactive=False)

            gr.Examples(
                examples=[
                    [
                        "Emotion Control",
                        "所有测试都顺利通过了，我们终于可以把这个好消息告诉大家！",
                        EXAMPLE_EMOTION_WAV,
                        "",
                        "高兴 (happy)",
                        "cantonese (广粤话 / Cantonese)",
                        "中速 (Normal)",
                        "中 (Medium)",
                        "中 (Medium)",
                    ],
                    [
                        "Dialect Control (Cantonese / Sichuanese)",
                        "今日天气几好，我哋食完饭一齐去海边行下啦。",
                        EXAMPLE_YUE_WAV,
                        "",
                        "高兴 (happy)",
                        "cantonese (广粤话 / Cantonese)",
                        "中速 (Normal)",
                        "中 (Medium)",
                        "中 (Medium)",
                    ],
                    [
                        "Dialect Control (Cantonese / Sichuanese)",
                        "今天天气巴适得很，我们吃完饭一起去河边耍一哈嘛。",
                        EXAMPLE_LINGGUANG_WAV,
                        EXAMPLE_LINGGUANG_TXT,
                        "高兴 (happy)",
                        "chuanyu dialect (川渝话 / Sichuanese)",
                        "中速 (Normal)",
                        "中 (Medium)",
                        "中 (Medium)",
                    ],
                    [
                        "Speed / Pitch / Volume (BASIC)",
                        "今晚的观测将在云层散开后开始，请耐心等待下一次通知。",
                        EXAMPLE_LINGGUANG_WAV,
                        EXAMPLE_LINGGUANG_TXT,
                        "高兴 (happy)",
                        "cantonese (广粤话 / Cantonese)",
                        "慢速 (Slow)",
                        "中 (Medium)",
                        "中 (Medium)",
                    ],
                ],
                inputs=[ctrl_mode, ctrl_text, ctrl_ref_audio, ctrl_ref_text, ctrl_emotion, ctrl_dialect, ctrl_speed, ctrl_pitch, ctrl_volume],
            )
            ctrl_btn.click(
                fn=ui_controlled_voice,
                inputs=[
                    model_selector, ctrl_mode, ctrl_text, ctrl_ref_audio, ctrl_ref_text,
                    ctrl_emotion, ctrl_dialect, ctrl_speed, ctrl_pitch, ctrl_volume,
                    ctrl_seed, ctrl_cfg, ctrl_sigma, ctrl_temp, ctrl_maxlen, ctrl_trim,
                ],
                outputs=[ctrl_audio_out, ctrl_file_out, ctrl_status],
            )

        # =====================================================================
        # TAB 3: Voice Design & Freeform Delivery Instructions
        # =====================================================================
        with gr.Tab("🎨 3. Voice Design & Freeform Delivery"):
            with gr.Row():
                with gr.Column(scale=6):
                    d_mode = gr.Radio(
                        choices=["Voice Design", "Freeform"],
                        value="Voice Design",
                        label="Instruction Mode (No Reference Audio Needed)",
                        info="Voice Design: describe who is speaking (age, gender, timbre, persona). Freeform: describe how the line is delivered.",
                    )
                    d_text = gr.Textbox(
                        label="Text to Synthesize",
                        lines=4,
                        value="The old radio crackled softly as the storm moved across the valley.",
                        info="Enter the text to speak. Long multi-sentence passages automatically lock onto the first sentence's generated voice!",
                    )
                    d_desc = gr.Textbox(
                        label="Voice Persona Description / Delivery Instruction",
                        lines=4,
                        value="Speak in a warm, intimate storytelling voice, with a gentle pace and a hint of nostalgia.",
                        info="Natural-language prompt in English or Chinese describing the voice or delivery, or build a 12-attribute sheet below.",
                    )
                    with gr.Accordion("🛠️ 12-Attribute Structured Voice Sheet Builder (Official Training Format)", open=False):
                        with gr.Row():
                            sh_gender = gr.Dropdown(["女", "男", "女童", "男童"], value="女", label="性别 (Gender)", info="Speaker gender.")
                            sh_age = gr.Dropdown(["青年", "中年", "老年", "少年", "儿童"], value="青年", label="年龄 (Age)", info="Speaker age bracket.")
                            sh_pitch = gr.Textbox(value="柔和中音，自然舒展", label="音高 (Pitch)", info="Pitch character.")
                        with gr.Row():
                            sh_speed = gr.Textbox(value="节奏从容，舒缓自然", label="语速 (Speed)", info="Pacing style.")
                            sh_vol = gr.Textbox(value="音量适中，贴近麦克风", label="音量 (Volume)", info="Projection level.")
                            sh_timbre = gr.Textbox(value="音色温暖清澈，略带磁性", label="音色质感 (Timbre)", info="Vocal texture.")
                        with gr.Row():
                            sh_emo = gr.Textbox(value="平静温柔，富有共情力", label="情绪 (Emotion)", info="Emotional tone.")
                            sh_inton = gr.Textbox(value="语调平稳柔和，尾音自然", label="语调 (Intonation)", info="Intonation contour.")
                            sh_pers = gr.Textbox(value="知性温和，亲切可信", label="性格 (Personality)", info="Persona character.")
                        sh_build_btn = gr.Button("Populate 12-Attribute Voice Design Sheet", size="sm")
                        sh_build_btn.click(
                            fn=ui_build_sheet,
                            inputs=[sh_gender, sh_age, sh_pitch, sh_speed, sh_vol, sh_timbre, sh_emo, sh_inton, sh_pers],
                            outputs=[d_desc, d_mode],
                        )

                    with gr.Accordion("🎙️ Optional Reference Timbre (For Freeform Mode Only)", open=False):
                        d_opt_ref = gr.Audio(
                            label="Optional Speaker Timbre Audio (Freeform mode only; leave empty for reference-free generation)",
                            type="filepath",
                        )

                    with gr.Accordion("⚙️ Advanced Sampling Settings", open=False):
                        with gr.Row():
                            d_seed = gr.Number(label="Seed", value=1895, precision=0, info="Random seed (-1 = random).")
                            d_cfg = gr.Slider(1.0, 5.0, value=2.0, step=0.1, label="CFG Scale", info="Flow-matching CFG scale.")
                            d_sigma = gr.Slider(0.0, 1.0, value=0.25, step=0.05, label="Sigma", info="Sampling noise scale.")
                        with gr.Row():
                            d_temp = gr.Slider(0.0, 1.0, value=0.0, step=0.05, label="Temperature", info="Sampling temperature.")
                            d_maxlen = gr.Slider(24, 120, value=80, step=4, label="Max Fragment Length (0.5B)", info="Keep high (80+) so style applies to whole sentence.")
                            d_trim = gr.Checkbox(value=True, label="Trim Trailing Silence", info="Trim trailing silence.")
                    d_btn = gr.Button("🎨 Synthesize Designed Voice", variant="primary")

                with gr.Column(scale=5):
                    d_audio_out = gr.Audio(label="Generated Speech (44.1 kHz)", type="numpy", interactive=False)
                    d_file_out = gr.File(label="Download .wav File")
                    d_status = gr.Textbox(label="Generation Log", lines=3, interactive=False)
                    with gr.Accordion("💾 Save Designed Voice to Drive Library (For 7-Language Cloning & Podcasts!)", open=True):
                        d_save_name = gr.Textbox(
                            label="Save Designed Voice As",
                            placeholder="e.g. designed_narrator_v1",
                            info="Saves this generated WAV + its text transcript to your Drive Voice Library so you can clone it in Tab 1 or Tab 5!",
                        )
                        d_save_btn = gr.Button("💾 Save Designed Voice to Drive Library", size="sm")
                        d_save_msg = gr.Textbox(label="Save Status", interactive=False, lines=1)

            gr.Examples(
                examples=[
                    [
                        "Voice Design",
                        "冰川在漫长岁月中缓缓移动，也悄悄记录着气候变化的痕迹。",
                        "成熟沉稳的男性纪录片旁白，吐字清晰，节奏舒缓，语气理性而有厚度。",
                    ],
                    [
                        "Voice Design",
                        "听……这个声音……是不是……很舒服……",
                        "这是一种ASMR耳语，属于一种旨在引发特殊感官体验的创意风格。这个女性使用轻柔的普通话进行耳语，声音气音成分重。音量极低，紧贴麦克风，语速极慢，旨在制造触发听者颅内快感的声学刺激。",
                    ],
                    [
                        "Voice Design",
                        "本宫赏你的，是恩典；本宫不给的，你断不能抢。在这后宫里，规矩重于天。",
                        "这是一个宫斗剧中的威严皇后的声音，展现了充满张力的戏剧叙事风格。她使用雍容华贵的普通话，中年女性声音沉稳。语速雍容和缓，但每个字都掷地有声，语气表面波澜不惊，实则暗藏锋芒和久居上位的威压。",
                    ],
                    [
                        "Freeform",
                        "Keep your eyes forward and your shoulders relaxed. One more round—you can do this!",
                        "Sound like an encouraging sports coach, energetic and firm without shouting.",
                    ],
                    [
                        "Freeform",
                        "The old radio crackled softly as the storm moved across the valley.",
                        "Speak in a warm, intimate storytelling voice, unhurried.",
                    ],
                ],
                inputs=[d_mode, d_text, d_desc],
            )
            d_btn.click(
                fn=ui_design_voice,
                inputs=[model_selector, d_text, d_desc, d_mode, d_opt_ref, d_seed, d_cfg, d_sigma, d_temp, d_maxlen, d_trim],
                outputs=[d_audio_out, d_file_out, d_status],
            )

        # =====================================================================
        # TAB 4: 482 Built-in Premium Voices (IP Voices)
        # =====================================================================
        with gr.Tab("🗣️ 4. Built-in Voices (482 IP Voices)"):
            with gr.Row():
                with gr.Column(scale=6):
                    b_cat = gr.Radio(
                        choices=VOICE_CATEGORY_CHOICES,
                        value="English (5)",
                        label="Filter Voice Catalog by Category",
                        info="Filter between the 5 English voices, 48 Chinese base voices, and 429 Chinese emotion-specific variants.",
                    )
                    b_voice = gr.Dropdown(
                        choices=VOICE_GROUPS.get("English", ALL_VOICES[:5]),
                        value="US_James",
                        allow_custom_value=True,
                        label="Built-in Voice ID",
                        info="Recommended English: US_James, US_Stella, US_Emma, UK_George, UK_Charlotte. Recommended Chinese: 灵小柔, 灵楷.",
                    )
                    b_persona_md = gr.Markdown(value=describe_builtin_voice("US_James"))
                    b_text = gr.Textbox(
                        label="Text to Synthesize",
                        lines=4,
                        value="Thanks for calling! Your order is on its way and should arrive tomorrow morning.",
                        info="Enter English or Chinese text to speak with the selected built-in voice.",
                    )
                    with gr.Accordion("⚙️ Advanced Sampling Settings", open=False):
                        with gr.Row():
                            b_seed = gr.Number(label="Seed", value=1895, precision=0, info="Random seed (-1 = random).")
                            b_cfg = gr.Slider(1.0, 5.0, value=2.0, step=0.1, label="CFG Scale", info="Flow-matching CFG scale.")
                            b_sigma = gr.Slider(0.0, 1.0, value=0.25, step=0.05, label="Sigma", info="Sampling noise scale.")
                        with gr.Row():
                            b_temp = gr.Slider(0.0, 1.0, value=0.0, step=0.05, label="Temperature", info="Sampling temperature.")
                            b_maxlen = gr.Slider(16, 80, value=50, step=2, label="Max Fragment Length (0.5B)", info="Clause split length.")
                            b_trim = gr.Checkbox(value=True, label="Trim Trailing Silence", info="Trim trailing silence.")
                    b_btn = gr.Button("🗣️ Speak with Built-in Voice", variant="primary")

                with gr.Column(scale=5):
                    b_audio_out = gr.Audio(label="Generated Speech (44.1 kHz)", type="numpy", interactive=False)
                    b_file_out = gr.File(label="Download .wav File")
                    b_status = gr.Textbox(label="Generation Log", lines=3, interactive=False)
                    with gr.Accordion("💾 Save Built-in Voice Sample to Drive Library (For 7-Language Cloning & Podcasts!)", open=False):
                        b_save_name = gr.Textbox(
                            label="Save Built-in Voice Sample As",
                            placeholder="e.g. ip_uk_george_ref",
                            info="Saves this generated IP voice WAV + transcript to your Drive Voice Library so you can clone it across all 7 languages or in podcasts!",
                        )
                        b_save_btn = gr.Button("💾 Save Built-in Voice to Drive Library", size="sm")
                        b_save_msg = gr.Textbox(label="Save Status", interactive=False, lines=1)

            b_cat.change(fn=filter_voice_choices, inputs=[b_cat], outputs=[b_voice, b_persona_md])
            b_voice.change(fn=describe_builtin_voice, inputs=[b_voice], outputs=[b_persona_md])
            b_btn.click(
                fn=ui_builtin_voice,
                inputs=[model_selector, b_text, b_voice, b_seed, b_cfg, b_sigma, b_temp, b_maxlen, b_trim],
                outputs=[b_audio_out, b_file_out, b_status],
            )

        # =====================================================================
        # TAB 5: Two-Speaker Podcast / Dialogue Studio
        # =====================================================================
        with gr.Tab("🎧 5. Two-Speaker Podcast Studio"):
            with gr.Row():
                with gr.Column(scale=6):
                    p_script = gr.Textbox(
                        label="Two-Speaker Dialogue Script (Start each line with 'speaker_1:' or 'speaker_2:')",
                        lines=6,
                        value=(
                            "speaker_1: 你最近是不是又去那家面馆了？\n"
                            "speaker_2: 对啊，排了半个小时的队，不过真的值得。\n"
                            "speaker_1: 下次也带上我吧，我也想尝尝招牌牛肉面！"
                        ),
                        info="On Ming-3.0-tts-tiny, runs native multi-speaker podcast generation. On Ming-3.0-tts-0.5B, automatically synthesizes & stitches each speaker turn (works on 16GB T4!).",
                    )
                    with gr.Row():
                        with gr.Column():
                            p_spk1_lib = gr.Dropdown(
                                choices=list_drive_voices(),
                                value="[Built-in Example] lingguang.wav",
                                label="📂 Load Speaker 1 from Library",
                                info="Pick a preset or saved voice from Drive.",
                            )
                            p_spk1_audio = gr.Audio(
                                label="Speaker 1 Reference Audio",
                                type="filepath",
                                value=EXAMPLE_LINGGUANG_WAV if os.path.exists(EXAMPLE_LINGGUANG_WAV) else None,
                            )
                            p_spk1_text = gr.Textbox(
                                label="Speaker 1 Reference Transcript",
                                lines=2,
                                value=EXAMPLE_LINGGUANG_TXT,
                                info="Transcript of Speaker 1's reference clip.",
                            )
                            p_spk1_asr = gr.Button("🪄 Auto-Transcribe Speaker 1", size="sm")
                        with gr.Column():
                            p_spk2_lib = gr.Dropdown(
                                choices=list_drive_voices(),
                                value="[Built-in Example] db30.wav",
                                label="📂 Load Speaker 2 from Library",
                                info="Pick a preset or saved voice from Drive.",
                            )
                            p_spk2_audio = gr.Audio(
                                label="Speaker 2 Reference Audio",
                                type="filepath",
                                value=EXAMPLE_DB30_WAV if os.path.exists(EXAMPLE_DB30_WAV) else None,
                            )
                            p_spk2_text = gr.Textbox(
                                label="Speaker 2 Reference Transcript",
                                lines=2,
                                value=EXAMPLE_DB30_TXT,
                                info="Transcript of Speaker 2's reference clip.",
                            )
                            p_spk2_asr = gr.Button("🪄 Auto-Transcribe Speaker 2", size="sm")
                    with gr.Row():
                        p_lang = gr.Dropdown(
                            choices=SUPPORTED_LANGUAGES,
                            value="Chinese",
                            label="Dialogue Language (for 0.5B Turn-Stitched Mode)",
                            info="Language used when synthesizing individual turns on Ming-3.0-tts-0.5B.",
                        )
                        p_pause = gr.Slider(
                            minimum=0.0,
                            maximum=1.0,
                            value=0.25,
                            step=0.05,
                            label="Inter-Turn Pause (seconds, 0.5B mode)",
                            info="Natural pause inserted between speaker turns when stitching on 0.5B.",
                        )
                        p_norm = gr.Checkbox(
                            value=True,
                            label="Peak-Balance Speaker Volumes",
                            info="Normalize peak levels across turns so both speakers match in loudness.",
                        )
                    with gr.Accordion("⚙️ Advanced Sampling Settings", open=False):
                        with gr.Row():
                            p_seed = gr.Number(label="Seed", value=1235, precision=0, info="Random seed (-1 = random).")
                            p_cfg = gr.Slider(1.0, 5.0, value=2.0, step=0.1, label="CFG Scale", info="Flow-matching CFG scale.")
                            p_sigma = gr.Slider(0.0, 1.0, value=0.25, step=0.05, label="Sigma", info="Sampling noise scale.")
                    p_btn = gr.Button("🎧 Generate Two-Speaker Podcast", variant="primary")

                with gr.Column(scale=5):
                    p_audio_out = gr.Audio(label="Generated Podcast (44.1 kHz)", type="numpy", interactive=False)
                    p_file_out = gr.File(label="Download Podcast .wav File")
                    p_status = gr.Textbox(label="Generation Log", lines=3, interactive=False)

            p_spk1_lib.change(fn=ui_load_drive_voice, inputs=[p_spk1_lib], outputs=[p_spk1_audio, p_spk1_text])
            p_spk2_lib.change(fn=ui_load_drive_voice, inputs=[p_spk2_lib], outputs=[p_spk2_audio, p_spk2_text])
            p_spk1_asr.click(fn=ui_transcribe_clip, inputs=[p_spk1_audio], outputs=[p_spk1_text])
            p_spk2_asr.click(fn=ui_transcribe_clip, inputs=[p_spk2_audio], outputs=[p_spk2_text])
            p_btn.click(
                fn=ui_podcast,
                inputs=[
                    model_selector, p_script, p_spk1_audio, p_spk1_text,
                    p_spk2_audio, p_spk2_text, p_lang, p_pause, p_norm, p_seed, p_cfg, p_sigma,
                ],
                outputs=[p_audio_out, p_file_out, p_status],
            )

            # Wire Drive Library save buttons (Tab 1, Tab 3 & Tab 4) to refresh all library dropdowns (Tab 1 & Tab 5)
            c_save_btn.click(
                fn=ui_save_drive_voice,
                inputs=[c_ref_audio, c_ref_text, c_save_name],
                outputs=[c_drive_voice, p_spk1_lib, p_spk2_lib, c_save_status],
            )
            d_save_btn.click(
                fn=ui_save_drive_voice,
                inputs=[d_file_out, d_text, d_save_name],
                outputs=[c_drive_voice, p_spk1_lib, p_spk2_lib, d_save_msg],
            )
            b_save_btn.click(
                fn=ui_save_drive_voice,
                inputs=[b_file_out, b_text, b_save_name],
                outputs=[c_drive_voice, p_spk1_lib, p_spk2_lib, b_save_msg],
            )

    demo.load(fn=get_system_welcome, inputs=None, outputs=[welcome_md])

demo.queue(concurrency_limit=1).launch(share=True, inline=True, show_error=True)


In [ ]:
#@title Step 5: Optional Keep-Alive Heartbeat (Prevent Colab Idle Disconnect)
import time

KEEP_ALIVE_MINUTES = 60 #@param {type:"slider", min:5, max:240, step:5}
RUN_KEEP_ALIVE = False #@param {type:"boolean"}

if RUN_KEEP_ALIVE:
    print(f"Keep-alive active for {KEEP_ALIVE_MINUTES} minutes. Press Stop to return to notebook cells.")
    try:
        for minute in range(1, int(KEEP_ALIVE_MINUTES) + 1):
            time.sleep(60)
            if minute % 5 == 0:
                print(f"[Keep-Alive] {minute}/{KEEP_ALIVE_MINUTES} min elapsed...")
    except KeyboardInterrupt:
        print("Keep-alive stopped by user.")
else:
    print("Keep-alive is disabled. Toggle RUN_KEEP_ALIVE=True and run this cell if leaving Gradio open.")


In [ ]:
#@title Step 6: One-Click Quick Test (Zero-Shot Cloning & Built-in Voice Verification)
import os
from IPython.display import Audio, FileLink, display

QUICKTEST_MODE = "Built-in Voice (US_James)" #@param ["Built-in Voice (US_James)", "Zero-Shot Voice Cloning (lingguang.wav)", "Voice Design (Storyteller)"]
QUICKTEST_TEXT = "Welcome back! [Sigh] After a long week of testing, all benchmarks passed on the first try." #@param {type:"string"}
QUICKTEST_SEED = 1895 #@param {type:"integer"}

try:
    if QUICKTEST_MODE == "Built-in Voice (US_James)":
        (sr, wav_i16), out_wav_path, log_msg = ENGINE.synthesize_builtin_voice(
            variant_label="0.5B",
            text=QUICKTEST_TEXT,
            voice_id="US_James",
            seed=QUICKTEST_SEED,
        )
    elif QUICKTEST_MODE == "Zero-Shot Voice Cloning (lingguang.wav)":
        (sr, wav_i16), out_wav_path, log_msg = ENGINE.synthesize_zero_shot(
            variant_label="0.5B",
            text=QUICKTEST_TEXT,
            ref_audio=EXAMPLE_LINGGUANG_WAV,
            ref_text=EXAMPLE_LINGGUANG_TXT,
            language="English",
            seed=QUICKTEST_SEED,
        )
    else:
        (sr, wav_i16), out_wav_path, log_msg = ENGINE.synthesize_design_or_freeform(
            variant_label="0.5B",
            text=QUICKTEST_TEXT,
            description="Speak in a warm, articulate documentary narrator voice, steady and engaging.",
            mode="Freeform",
            seed=QUICKTEST_SEED,
        )

    print(log_msg)
    display(Audio(filename=out_wav_path, rate=sr))
    display(FileLink(out_wav_path))
except Exception as e:
    print(f"Quick test failed: {e}")
    raise


In [ ]:
#@title Step 7: Batch Script / Audiobook Synthesis to Google Drive
import os
import csv
from tqdm.auto import tqdm
from IPython.display import FileLink, display

BATCH_LINES_OR_TXT_PATH = "Welcome to chapter one of our audio guide.\n[Breathing] In this section, we explore how neural flow-matching synthesizes studio 44.1 kHz speech.\nThanks for listening, and see you in the next episode!" #@param {type:"string"}
BATCH_VOICE_MODE = "Built-in Voice (IP)" #@param ["Built-in Voice (IP)", "Zero-Shot Clone (Reference Audio)"]
BATCH_BUILTIN_VOICE = "US_Emma" #@param {type:"string"}
BATCH_REF_AUDIO = "/content/Ming-3.0-tts/data/voice_presets/voice_audio/lingguang.wav" #@param {type:"string"}
BATCH_REF_TEXT = "诶，你今天有空吗？我们能不能聊聊这件事啊？" #@param {type:"string"}
BATCH_LANGUAGE = "English" #@param ["Chinese", "English", "Japanese", "Korean", "German", "French", "Spanish"]
BATCH_SEED = 1895 #@param {type:"integer"}

batch_out_dir = os.path.join(OUTPUT_DIR, "batch_runs")
os.makedirs(batch_out_dir, exist_ok=True)

if os.path.isfile(BATCH_LINES_OR_TXT_PATH):
    with open(BATCH_LINES_OR_TXT_PATH, "r", encoding="utf-8") as bf:
        lines = [ln.strip() for ln in bf.read().splitlines() if ln.strip()]
else:
    lines = [ln.strip() for ln in BATCH_LINES_OR_TXT_PATH.replace("\\n", "\n").splitlines() if ln.strip()]

manifest_path = os.path.join(batch_out_dir, "ming3_batch_manifest.csv")
results = []

print(f"Starting batch synthesis of {len(lines)} item(s) -> {batch_out_dir} ...")
for idx, line_text in enumerate(tqdm(lines, desc="Ming 3.0 TTS Batch")):
    try:
        if BATCH_VOICE_MODE == "Built-in Voice (IP)":
            _, wav_path, status = ENGINE.synthesize_builtin_voice(
                variant_label="0.5B",
                text=line_text,
                voice_id=BATCH_BUILTIN_VOICE,
                seed=BATCH_SEED + idx,
            )
        else:
            _, wav_path, status = ENGINE.synthesize_zero_shot(
                variant_label="0.5B",
                text=line_text,
                ref_audio=BATCH_REF_AUDIO,
                ref_text=BATCH_REF_TEXT,
                language=BATCH_LANGUAGE,
                seed=BATCH_SEED + idx,
            )
        results.append({"index": idx + 1, "status": "OK", "file": wav_path, "text": line_text})
        display(FileLink(wav_path))
    except Exception as item_err:
        print(f"Item {idx + 1} failed: {item_err}")
        results.append({"index": idx + 1, "status": f"ERROR: {item_err}", "file": "", "text": line_text})

with open(manifest_path, "w", newline="", encoding="utf-8") as mf:
    writer = csv.DictWriter(mf, fieldnames=["index", "status", "file", "text"])
    writer.writeheader()
    writer.writerows(results)

print(f"Batch complete! Manifest saved to: {manifest_path}")
display(FileLink(manifest_path))
